# 01 — Create Benchmark Datasets: Lance vs Parquet

**Purpose:** Extract frames from BDD100K dashcam videos and write to two storage formats at three dataset sizes — to support the Lance vs Parquet throughput benchmark in `02_training_benchmark.ipynb`.

| Format | Storage | Schema |
|--------|---------|--------|
| **Lance** | Inline JPEG bytes — blob layout in UC Volume | `image` as `LARGE_BINARY` |
| **Delta/Parquet** | JPEG files in UC Volume; metadata in UC table | `image_path` as `STRING` |

Run this notebook once per size tier (`10k`, `100k`, `1m`) using the `size` widget.

## What this notebook does

1. **Configure paths.** Resolve UC namespace and Volume paths from widgets.
2. **Load BDD100K annotations.** Parse the detection JSON into a `video_id → bboxes` lookup.
3. **Extract frames in parallel.** Ray workers use `decord` to decode frames at the target fps, encode as JPEG, and simultaneously write to Lance (inline bytes) and to Volumes (JPEG files for the Parquet tier).
4. **Write Delta table.** Collect metadata rows from workers; write to a UC Delta table with `image_path` references.
5. **Verify.** Confirm row counts match the target size for both formats.

---

**Inputs:**
- BDD100K video files in a UC Volume (`.mov` / `.mp4`)
- BDD100K detection annotations JSON (`det_train.json`)

**Outputs (per size tier):**
- Lance dataset at `/Volumes/{catalog}/{schema}/{volume}/bdd100k_lance_{size}/`
- JPEG image files at `/Volumes/{catalog}/{schema}/{volume}/bdd100k_parquet_{size}/`
- Delta UC table `{catalog}.{schema}.bdd100k_parquet_{size}`

**Next:** `02_training_benchmark.ipynb`

In [ ]:
# Install packages BEFORE any ray.init or @ray_launch call — installing after shuts down the cluster
%pip install -qU ray[all]==2.54.0 lance==0.17.0 decord Pillow pyarrow>=16.0 databricks-sdk>=0.49.0
# Install serverless_gpu from your Workspace — update the path to match your environment
# %pip install '/Workspace/Users/<your-user>/ray-on-databricks-rct/distributed-training/XGBoost/databricks.serverless_gpu-0.5.3-py3-none-any.whl'
dbutils.library.restartPython()

In [ ]:
# ── Widgets ────────────────────────────────────────────────────────────────
dbutils.widgets.dropdown("size",             "10k", ["10k", "100k", "1m"], "Dataset Size")
dbutils.widgets.text("catalog",              "main",         "UC Catalog")
dbutils.widgets.text("schema",               "ml_benchmark", "UC Schema")
dbutils.widgets.text("volume",               "bdd100k",      "UC Volume")
dbutils.widgets.text("videos_path",          "",             "BDD100K Videos Path (leave blank for default)")
dbutils.widgets.text("annotations_path",     "",             "BDD100K Annotations JSON (leave blank for default)")
dbutils.widgets.text("fps",                  "1",            "Frame Sampling Rate (fps)")

size    = dbutils.widgets.get("size")
catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")
volume  = dbutils.widgets.get("volume")
fps     = int(dbutils.widgets.get("fps"))

SIZE_MAP = {"10k": 10_000, "100k": 100_000, "1m": 1_000_000}
n_frames = SIZE_MAP[size]

base_vol        = f"/Volumes/{catalog}/{schema}/{volume}"
videos_path     = dbutils.widgets.get("videos_path")      or f"{base_vol}/videos"
annotations_path = dbutils.widgets.get("annotations_path") or f"{base_vol}/annotations/det_train.json"

lance_path       = f"{base_vol}/bdd100k_lance_{size}"
parquet_img_dir  = f"{base_vol}/bdd100k_parquet_{size}"
delta_table      = f"{catalog}.{schema}.bdd100k_parquet_{size}"

print(f"Size tier       : {size} ({n_frames:,} frames)")
print(f"Videos path     : {videos_path}")
print(f"Annotations     : {annotations_path}")
print(f"Lance output    : {lance_path}")
print(f"Parquet images  : {parquet_img_dir}")
print(f"Delta table     : {delta_table}")

In [ ]:
import os

# Set credentials so Ray workers can authenticate to Databricks services
os.environ['DATABRICKS_HOST']  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ['DATABRICKS_TOKEN'] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

In [ ]:
# Ensure Ray checkpoint/tmp volume exists
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import catalog as sdk_catalog

w = WorkspaceClient()
ray_tmp_path = f"/Volumes/{catalog}/{schema}/ray_tmp"

if not os.path.exists(ray_tmp_path):
    w.volumes.create(
        catalog_name=catalog,
        schema_name=schema,
        name="ray_tmp",
        volume_type=sdk_catalog.VolumeType.MANAGED,
    )
    print(f"Created volume: {ray_tmp_path}")
else:
    print(f"Volume exists : {ray_tmp_path}")

# Also ensure output directories exist on the driver before workers write to them
os.makedirs(parquet_img_dir, exist_ok=True)
print(f"Parquet image dir ready: {parquet_img_dir}")

## Frame extraction worker

Each `extract_video_frames` task handles one video:
1. Decodes frames at the target fps using `decord` (CPU)
2. Encodes each frame as JPEG bytes
3. Writes the JPEG file to the Parquet images directory in Volumes
4. Appends a `pyarrow.Table` batch to the Lance dataset (inline bytes, blob layout)
5. Returns a list of metadata dicts (no image bytes) for the Delta table

The Lance schema is defined locally inside each task to avoid PyArrow serialization through Ray's object store.

In [ ]:
import ray

# Capture closure variables — only small scalars/strings are captured, not large objects
_lance_path      = lance_path
_parquet_img_dir = parquet_img_dir
_fps             = fps


@ray.remote(num_cpus=2)
def extract_video_frames(video_path, bboxes_for_video):
    """Extract frames from one video. Writes to Lance (inline bytes) and Volumes (JPEG files).
    Returns metadata rows for the Delta table (no image bytes)."""
    import io, os, uuid, json, math
    import pyarrow as pa
    import lance
    import decord
    from PIL import Image
    from datetime import datetime, timezone

    video_id = os.path.splitext(os.path.basename(video_path))[0]
    os.makedirs(_parquet_img_dir, exist_ok=True)

    try:
        vr = decord.VideoReader(video_path, ctx=decord.cpu(0))
        video_fps = float(vr.get_avg_fps())
    except Exception as e:
        print(f"Skipping {video_id}: {e}")
        return []

    frame_step   = max(1, round(video_fps / _fps))
    frame_indices = list(range(0, len(vr), frame_step))

    # Lance schema — defined locally to avoid serialization through Ray object store
    bbox_struct = pa.struct([
        pa.field("category", pa.string()),
        pa.field("x1", pa.float32()), pa.field("y1", pa.float32()),
        pa.field("x2", pa.float32()), pa.field("y2", pa.float32()),
    ])
    lance_schema = pa.schema([
        pa.field("frame_id",     pa.string()),
        pa.field("video_id",     pa.string()),
        pa.field("frame_number", pa.int32()),
        pa.field("timestamp_ms", pa.int64()),
        pa.field("image",        pa.large_binary()),
        pa.field("width",        pa.int32()),
        pa.field("height",       pa.int32()),
        pa.field("bbox_labels",  pa.list_(bbox_struct)),
        pa.field("extracted_at", pa.timestamp("us", tz="UTC")),
    ])

    lance_rows, metadata_rows = [], []

    for fidx in frame_indices:
        frame_arr = vr[fidx].asnumpy()
        h, w = frame_arr.shape[:2]

        img = Image.fromarray(frame_arr)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=85)
        jpeg_bytes = buf.getvalue()

        frame_id = str(uuid.uuid4())
        ts_ms    = int(fidx / video_fps * 1000)
        ext_at   = datetime.now(timezone.utc)
        img_path = os.path.join(_parquet_img_dir, f"{frame_id}.jpg")

        with open(img_path, "wb") as f:
            f.write(jpeg_bytes)

        lance_rows.append({
            "frame_id":     frame_id,
            "video_id":     video_id,
            "frame_number": fidx,
            "timestamp_ms": ts_ms,
            "image":        jpeg_bytes,
            "width":        w,
            "height":       h,
            "bbox_labels":  bboxes_for_video,
            "extracted_at": ext_at,
        })

        metadata_rows.append({
            "frame_id":     frame_id,
            "video_id":     video_id,
            "frame_number": fidx,
            "timestamp_ms": ts_ms,
            "image_path":   img_path,
            "width":        w,
            "height":       h,
            "bbox_labels":  json.dumps(bboxes_for_video),
            "extracted_at": ext_at.isoformat(),
        })

    # Append this video's frames to Lance (each append creates an independent fragment)
    if lance_rows:
        table = pa.Table.from_pylist(lance_rows, schema=lance_schema)
        lance.write_dataset(table, _lance_path, mode="append", schema=lance_schema)

    return metadata_rows

## DatasetBuilder actor + @ray_launch

The `DatasetBuilder` actor runs on one worker node (via `@ray.remote(num_cpus=1)`). It:
1. Loads BDD100K annotations from the Volumes path (avoids passing the large annotation dict as a closure)
2. Selects enough videos to produce the target frame count (~40 frames/video at 1fps, +20% buffer)
3. Launches one `extract_video_frames` task per video in parallel across the Ray cluster
4. Collects metadata rows and trims to the exact target count

The `@ray_launch` decorator provisions a multi-node Serverless GPU cluster. The `TaskRunner` is scheduled on a worker, not the head node.

In [ ]:
# Capture only small scalars as closures — large data (annotations) is loaded inside the worker
_annot_path_cl  = annotations_path
_videos_path_cl = videos_path
_n_frames_cl    = n_frames
_fps_cl         = fps


@ray.remote(num_cpus=1)
class DatasetBuilder:
    def run(self):
        import os, json, math
        import ray

        # Load annotations on the worker — avoids serializing a large dict through the closure
        with open(_annot_path_cl) as f:
            raw_ann = json.load(f)
        annotations = {}
        for item in raw_ann:
            vid_id = item["name"].rsplit(".", 1)[0]
            bboxes = []
            for lbl in (item.get("labels") or []):
                if "box2d" in lbl:
                    bboxes.append({
                        "category": lbl["category"],
                        "x1": float(lbl["box2d"]["x1"]),
                        "y1": float(lbl["box2d"]["y1"]),
                        "x2": float(lbl["box2d"]["x2"]),
                        "y2": float(lbl["box2d"]["y2"]),
                    })
            annotations[vid_id] = bboxes
        print(f"Loaded annotations for {len(annotations):,} videos")

        # List and select videos to process
        video_files = sorted([
            os.path.join(_videos_path_cl, f)
            for f in os.listdir(_videos_path_cl)
            if f.lower().endswith((".mov", ".mp4"))
        ])
        avg_frames_per_video = 40 * _fps_cl  # BDD100K: ~40s/video
        n_videos = min(len(video_files), math.ceil(_n_frames_cl / avg_frames_per_video * 1.2))
        selected = video_files[:n_videos]
        print(f"Selected {n_videos:,} videos to extract {_n_frames_cl:,} frames")

        # Launch parallel extraction — one task per video
        futures = [
            extract_video_frames.remote(
                vp,
                annotations.get(os.path.splitext(os.path.basename(vp))[0], []),
            )
            for vp in selected
        ]

        # Collect in order; stop accumulating once target is met
        all_metadata = []
        for rows in ray.get(futures):
            all_metadata.extend(rows)
            if len(all_metadata) >= _n_frames_cl:
                break

        trimmed = all_metadata[:_n_frames_cl]
        print(f"Returning {len(trimmed):,} metadata rows")
        return trimmed

In [ ]:
from serverless_gpu.ray import ray_launch


@ray_launch(gpus=4, gpu_type='A10', remote=True)
def build_benchmark_datasets():
    builder = DatasetBuilder.remote()
    return ray.get(builder.run.remote())


metadata_rows = build_benchmark_datasets.distributed()
print(f"\nExtracted {len(metadata_rows):,} frames")

## Write Delta table (Parquet metadata)

The metadata rows returned from Ray workers contain `image_path` strings pointing to JPEG files already written to Volumes. We write these to a Delta UC table so `ray.data.read_databricks_tables` can query them in the benchmark notebook.

In [ ]:
import json
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType
)

delta_schema = StructType([
    StructField("frame_id",     StringType(),  False),
    StructField("video_id",     StringType(),  False),
    StructField("frame_number", IntegerType(), False),
    StructField("timestamp_ms", LongType(),    False),
    StructField("image_path",   StringType(),  False),
    StructField("width",        IntegerType(), False),
    StructField("height",       IntegerType(), False),
    StructField("bbox_labels",  StringType(),  True),  # JSON string
    StructField("extracted_at", StringType(),  False),
])

rows = [Row(**r) for r in metadata_rows]
df   = spark.createDataFrame(rows, schema=delta_schema)

df.write.format("delta").mode("overwrite").saveAsTable(delta_table)
print(f"Written {df.count():,} rows to {delta_table}")

## Verification

Confirm both formats contain the expected row count for this size tier.

In [ ]:
import lance

# Delta row count
delta_count = spark.sql(f"SELECT COUNT(*) AS n FROM {delta_table}").collect()[0]["n"]

# Lance row count
lance_count = lance.dataset(lance_path).count_rows()

print(f"Target frames : {n_frames:,}")
print(f"Delta rows    : {delta_count:,}  {'✓' if delta_count == n_frames else '✗'}")
print(f"Lance rows    : {lance_count:,}  {'✓' if lance_count == n_frames else '✗'}")

# Quick schema sanity check
print("\nDelta schema:")
spark.sql(f"DESCRIBE {delta_table}").show(truncate=False)

print("\nLance schema:")
print(lance.dataset(lance_path).schema)